<a href="https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Fluidos_y_Medios_Continuos/Fluidos_Navier_Stokes.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



# Investigación Avanzada: Fluidos Navier Stokes

Simulación numérica de vanguardia correspondiente a los Problemas Abiertos y Fronteras de Investigación (Nivel Doctorado).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def simulate_lbm_cylinder():
    # Lattice Boltzmann Method (D2Q9)
    maxIter = 5000  # Total number of time iterations.
    Re = 150.0      # Reynolds number.
    nx, ny = 400, 100 # Numer of lattice nodes.
    ly = ny-1
    cx, cy, r = nx//4, ny//2, 10 # Cylinder coordinates and radius.
    uLB     = 0.04                  # Velocity in lattice units.
    nulb    = uLB*r/Re;             # Viscoscity in lattice units.
    omega = 1 / (3*nulb+0.5);    # Relaxation parameter.
    
    # Lattice constants
    v = np.array([ [ 1,  1], [ 1,  0], [ 1, -1], [ 0,  1], [ 0,  0],
                   [ 0, -1], [-1,  1], [-1,  0], [-1, -1] ])
    t = np.array([ 1/36, 1/9, 1/36, 1/9, 4/9, 1/9, 1/36, 1/9, 1/36])
    
    col1 = np.array([0, 1, 2])
    col2 = np.array([3, 4, 5])
    col3 = np.array([6, 7, 8])
    
    # Macroscopic variables
    def macroscopic(fin):
        rho = np.sum(fin, axis=0)
        u = np.zeros((2, nx, ny))
        for i in range(9):
            u[0,:,:] += v[i,0] * fin[i,:,:]
            u[1,:,:] += v[i,1] * fin[i,:,:]
        u /= rho
        return rho, u
        
    # Equilibrium distribution function
    def equilibrium(rho, u):
        usqr = 3/2 * (u[0]**2 + u[1]**2)
        feq = np.zeros((9,nx,ny))
        for i in range(9):
            cu = 3 * (v[i,0]*u[0,:,:] + v[i,1]*u[1,:,:])
            feq[i,:,:] = rho*t[i] * (1 + cu + 0.5*cu**2 - usqr)
        return feq
        
    # Setup cylinder boundary
    obstacle = np.fromfunction(lambda x,y: (x-cx)**2+(y-cy)**2<r**2, (nx,ny))
    
    # Initial conditions
    vel = np.array([np.full((nx,ny), uLB), np.zeros((nx,ny))])
    fin = equilibrium(np.ones((nx,ny)), vel)
    
    # Main loop
    for time in range(maxIter):
        # Right boundary condition: outflow
        fin[col3, -1, :] = fin[col3, -2, :] 
        
        # Compute macroscopic variables
        rho, u = macroscopic(fin)
        
        # Left boundary condition: inflow
        u[:,0,:] = vel[:,0,:]
        rho[0,:] = 1/(1-u[0,0,:]) * (np.sum(fin[col2,0,:], axis=0) + 
                                     2*np.sum(fin[col3,0,:], axis=0))
        feq = equilibrium(rho, u)
        fin[col1,0,:] = feq[col1,0,:] + fin[col3,0,:] - feq[col3,0,:]
        
        # Collision step
        fout = fin - omega * (fin - feq)
        
        # Bounce-back condition for obstacle
        for i in range(9):
            fout[i, obstacle] = fin[8-i, obstacle]
            
        # Streaming step
        for i in range(9):
            fin[i,:,:] = np.roll(np.roll(fout[i,:,:], v[i,0], axis=0), v[i,1], axis=1)
            
    # Plotting the final velocity field
    rho, u = macroscopic(fin)
    velocity = np.sqrt(u[0]**2+u[1]**2)
    velocity[obstacle] = 0
    
    plt.figure(figsize=(10, 4))
    plt.imshow(velocity.T, cmap='jet')
    plt.colorbar(label='Velocity')
    plt.title('Von Karman Vortex Street (Navier-Stokes via LBM)')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('Fluidos_Navier_Stokes.png')
    plt.show()

if __name__ == '__main__':
    simulate_lbm_cylinder()
